In [1]:
# ---------------------------------------------------------
# Cell 1: Import Libraries
# ---------------------------------------------------------
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import ipywidgets as widgets
from IPython.display import display, clear_output
import urllib.request
import zipfile
import os
import warnings
warnings.filterwarnings('ignore') # Keeps our presentation clean from red warning text

print("Step 1 Complete: All libraries imported successfully!")

Step 1 Complete: All libraries imported successfully!


In [2]:
# ---------------------------------------------------------
# Cell 2: Download and Load the MovieLens Dataset
# ---------------------------------------------------------
def fetch_movielens_data():
    """Downloads and extracts the dataset if it doesn't already exist."""
    url = "https://files.grouplens.org/datasets/movielens/ml-100k.zip"
    zip_path = "ml-100k.zip"
    
    if not os.path.exists("ml-100k"):
        print("Downloading MovieLens 100k dataset (this only happens once)...")
        urllib.request.urlretrieve(url, zip_path)
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall()
        print("Download complete.")
    else:
        print("Dataset found locally.")

fetch_movielens_data()

# Read the data files into pandas DataFrames
ratings = pd.read_csv('ml-100k/u.data', sep='\t', names=['user_id', 'movie_id', 'rating', 'timestamp'], encoding='latin-1')
users = pd.read_csv('ml-100k/u.user', sep='|', names=['user_id', 'age', 'gender', 'occupation', 'zip_code'], encoding='latin-1')
movie_cols = ['movie_id', 'title', 'release_date', 'video_release_date', 'imdb_url', 'unknown', 'Action', 'Adventure', 
              'Animation', 'Children', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 
              'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']
movies = pd.read_csv('ml-100k/u.item', sep='|', names=movie_cols, encoding='latin-1')

# Combine them into one master dataset
data = pd.merge(pd.merge(ratings, users, on='user_id'), movies, on='movie_id')

print(f"Step 2 Complete: Loaded {len(data)} real-world movie ratings!")

Dataset found locally.
Step 2 Complete: Loaded 100000 real-world movie ratings!


In [3]:
# ---------------------------------------------------------
# Cell 3: Model Training and Preprocessing
# ---------------------------------------------------------
print("Preparing data and training the model...")

# 1. Feature Engineering: Convert text to numbers
data['gender_num'] = data['gender'].map({'M': 1, 'F': 0})

# Calculate averages to give the model context about users and movies
user_avg = data.groupby('user_id')['rating'].mean().reset_index().rename(columns={'rating': 'user_avg_rating'})
movie_avg = data.groupby('movie_id')['rating'].mean().reset_index().rename(columns={'rating': 'movie_avg_rating'})

data = pd.merge(data, user_avg, on='user_id')
data = pd.merge(data, movie_avg, on='movie_id')

# 2. Define the exact features we want the model to learn from
features = ['user_id', 'movie_id', 'age', 'gender_num', 'user_avg_rating', 'movie_avg_rating', 
            'Action', 'Comedy', 'Drama', 'Sci-Fi', 'Romance']

X = data[features]
y = data['rating']

# 3. Train the model
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=100, learning_rate=0.1, max_depth=5)
model.fit(X_train, y_train)

# Evaluate
predictions = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, predictions))

# 4. Prepare a "Movie Catalog" DataFrame for our live UI to use later
# This contains just the unique movies and their specific features
movie_catalog = data[['movie_id', 'title', 'movie_avg_rating', 'Action', 'Comedy', 'Drama', 'Sci-Fi', 'Romance']].drop_duplicates()

print(f"Step 3 Complete! Model trained successfully.")
print(f"Error margin (RMSE): {rmse:.2f} stars.")

Preparing data and training the model...
Step 3 Complete! Model trained successfully.
Error margin (RMSE): 0.92 stars.


In [2]:
# ---------------------------------------------------------
# Cell 4: Interactive Live-Inference Profile Builder
# ---------------------------------------------------------
import ipywidgets as widgets
from IPython.display import display, clear_output

# --- 1. Define the UI Sliders and Buttons ---
age_slider = widgets.IntSlider(value=30, min=10, max=90, description='Age:')
gender_dropdown = widgets.Dropdown(options=[('Female', 0), ('Male', 1)], description='Gender:')
critic_slider = widgets.FloatSlider(value=3.5, min=1.0, max=5.0, step=0.1, description='Strictness:', 
                                    tooltip='Lower = Hates everything, Higher = Loves everything')
predict_btn = widgets.Button(description="Predict My Top 5 Movies!", button_style='primary')
output_area = widgets.Output()

# --- 2. The Prediction Logic ---
def generate_live_predictions(b):
    with output_area:
        clear_output()
        print("Analyzing your unique profile against 1,600+ movies...\n")
        
        # Make a copy of our movie catalog to fill with the user's custom info
        custom_user_data = movie_catalog.copy()
        
        # Apply the user's slider choices to every movie row
        custom_user_data['user_id'] = 999999 # A dummy ID for the live user
        custom_user_data['age'] = age_slider.value
        custom_user_data['gender_num'] = gender_dropdown.value
        custom_user_data['user_avg_rating'] = critic_slider.value
        
        # Ensure the columns match the exact order the model was trained on
        X_live = custom_user_data[features]
        
        # Predict the ratings!
        custom_user_data['predicted_rating'] = model.predict(X_live)
        
        # Sort to find the highest predicted ratings
        top_5 = custom_user_data.sort_values('predicted_rating', ascending=False).head(5)
        
        print("🍿 Based on your profile, you should watch:")
        for index, row in top_5.iterrows():
            stars = round(row['predicted_rating'], 2)
            print(f"⭐️ {stars} / 5.0 --> {row['title']}")

# --- 3. Connect and Display ---
predict_btn.on_click(generate_live_predictions)

ui_layout = widgets.VBox([
    widgets.HTML("<h2>🎬 AI Movie Matchmaker</h2>"),
    widgets.HTML("<p>Adjust the sliders to build a profile. The XGBoost model will instantly predict your favorite movies!</p>"),
    age_slider,
    gender_dropdown,
    critic_slider,
    widgets.HTML("<br>"),
    predict_btn,
    widgets.HTML("<hr>"),
    output_area
])

display(ui_layout)